# 04 — Dataset Consolidation

This notebook consolidates the raw INMET files for 2018–2024.

Because the original files contain meteorological observations from multiple stations and total several gigabytes, they are processed in chunks.

The pipeline:

- scans all raw CSV files;
- reads each file incrementally;
- keeps only the Recife automatic weather station (WMO A301);
- selects only the variables required by the project;
- validates station identity;
- combines observations from 2018–2024;
- sorts the records chronologically;
- removes duplicated timestamps;
- saves a compact Recife-specific dataset for subsequent preprocessing.

In [2]:
from pathlib import Path
import re
import unicodedata

import pandas as pd


# ============================================================
# 1. PATHS AND CONFIGURATION
# ============================================================

RAW_DIR = Path("../data/raw")

OUTPUT_PATH = Path(
    "../data/processed/recife_2018_2024_filtered.csv"
)

STATION_CODE = "A301"

START_DATE = pd.Timestamp(
    "2018-01-01",
    tz="UTC"
)

END_DATE = pd.Timestamp(
    "2024-12-31 23:59:59",
    tz="UTC"
)

CHUNK_SIZE = 200_000


# ============================================================
# 2. COLUMNS REQUIRED BY THE PROJECT
# ============================================================

required_columns = [
    "Data",
    "HORA (UTC)",
    "PRECIPITAÇÃO TOTAL, HORÁRIO (mm)",
    "PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)",
    "PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)",
    "PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)",
    "TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)",
    "TEMPERATURA DO PONTO DE ORVALHO (°C)",
    "TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)",
    "TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)",
    "TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C)",
    "TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C)",
    "UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)",
    "UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)",
    "UMIDADE RELATIVA DO AR, HORARIA (%)",
    "VENTO, DIREÇÃO HORARIA (gr) (° (gr))",
    "VENTO, RAJADA MAXIMA (m/s)",
    "VENTO, VELOCIDADE HORARIA (m/s)",
    "ESTACAO",
    "CODIGO (WMO)",
    "UF",
    "LATITUDE",
    "LONGITUDE",
]


# ============================================================
# 3. NORMALIZE COLUMN NAMES FOR COMPARISON
# ============================================================

def normalize_column_name(name):
    """
    Creates a simplified representation of a column name.

    Example:
    'HORA (UTC)'
    'Hora UTC'
    'HORA UTC'

    all become approximately:
    'HORAUTC'
    """

    name = str(name).strip().replace("\n", "")

    name = unicodedata.normalize(
        "NFKD",
        name
    )

    name = "".join(
        character
        for character in name
        if not unicodedata.combining(character)
    )

    name = name.upper()

    name = re.sub(
        r"[^A-Z0-9]",
        "",
        name
    )

    return name


canonical_columns = {
    normalize_column_name(column): column
    for column in required_columns
}


def standardize_columns(columns):
    """
    Converts header variations to the canonical names
    used by this project.
    """

    mapping = {}

    for original_column in columns:

        normalized = normalize_column_name(
            original_column
        )

        if normalized in canonical_columns:

            mapping[original_column] = (
                canonical_columns[normalized]
            )

    return mapping


# ============================================================
# 4. FIND RAW FILES
# ============================================================

raw_files = sorted(
    RAW_DIR.glob("*.csv")
)

if not raw_files:
    raise FileNotFoundError(
        f"No CSV files found in {RAW_DIR.resolve()}"
    )

print(
    f"Raw files found: {len(raw_files)}"
)

for file in raw_files:
    print("-", file.name)


# ============================================================
# 5. DETECT CSV FORMAT
# ============================================================

def detect_csv_format(file_path):

    encodings = [
        "utf-8",
        "utf-8-sig",
        "latin-1",
    ]

    separators = [
        ",",
        ";",
    ]

    station_column_key = normalize_column_name(
        "CODIGO (WMO)"
    )

    for encoding in encodings:

        for separator in separators:

            try:

                header = pd.read_csv(
                    file_path,
                    sep=separator,
                    encoding=encoding,
                    nrows=0,
                )

                normalized_header = {
                    normalize_column_name(column)
                    for column in header.columns
                }

                if station_column_key in normalized_header:

                    return encoding, separator

            except Exception:
                continue

    raise ValueError(
        f"Could not determine CSV format for "
        f"{file_path.name}"
    )


# ============================================================
# 6. PROCESS FILES IN CHUNKS
# ============================================================

recife_chunks = []

total_rows_read = 0
total_recife_rows = 0


for file_number, file_path in enumerate(
    raw_files,
    start=1,
):

    print()
    print(
        f"Processing file "
        f"{file_number}/{len(raw_files)}: "
        f"{file_path.name}"
    )

    encoding, separator = detect_csv_format(
        file_path
    )

    print(
        f"Encoding: {encoding}"
    )

    print(
        f"Separator: {repr(separator)}"
    )


    # --------------------------------------------------------
    # Read only header first
    # --------------------------------------------------------

    header = pd.read_csv(
        file_path,
        sep=separator,
        encoding=encoding,
        nrows=0,
    )


    # --------------------------------------------------------
    # Match raw column names to canonical project names
    # --------------------------------------------------------

    rename_map = standardize_columns(
        header.columns
    )

    canonical_found = set(
        rename_map.values()
    )

    missing_columns = [
        column
        for column in required_columns
        if column not in canonical_found
    ]

    if missing_columns:

        print()
        print(
            "Columns available in file:"
        )

        for column in header.columns:
            print("-", repr(column))

        raise ValueError(
            f"{file_path.name} is missing columns: "
            f"{missing_columns}"
        )


    # --------------------------------------------------------
    # Only read columns required by project
    # --------------------------------------------------------

    original_columns_to_read = list(
        rename_map.keys()
    )

    reader = pd.read_csv(
        file_path,
        sep=separator,
        encoding=encoding,
        usecols=original_columns_to_read,
        chunksize=CHUNK_SIZE,
        low_memory=False,
    )


    # --------------------------------------------------------
    # Process file incrementally
    # --------------------------------------------------------

    for chunk in reader:

        total_rows_read += len(chunk)

        chunk = chunk.rename(
            columns=rename_map
        )

        chunk["CODIGO (WMO)"] = (
            chunk["CODIGO (WMO)"]
            .astype(str)
            .str.strip()
        )

        recife_chunk = chunk[
            chunk["CODIGO (WMO)"]
            == STATION_CODE
        ].copy()

        if recife_chunk.empty:
            continue

        total_recife_rows += len(
            recife_chunk
        )

        recife_chunks.append(
            recife_chunk
        )


# ============================================================
# 7. CONCATENATE RECIFE OBSERVATIONS
# ============================================================

if not recife_chunks:
    raise ValueError(
        f"No observations found for "
        f"station {STATION_CODE}."
    )

df = pd.concat(
    recife_chunks,
    ignore_index=True,
)

print()
print(
    "Rows read from original files:",
    total_rows_read
)

print(
    "Recife rows collected:",
    total_recife_rows
)


# ============================================================
# 8. CREATE DATETIME
# ============================================================

df["datetime_utc"] = pd.to_datetime(
    df["Data"].astype(str).str.strip()
    + " "
    + df["HORA (UTC)"].astype(str).str.strip(),
    errors="coerce",
    utc=True,
)

invalid_datetime = (
    df["datetime_utc"]
    .isna()
    .sum()
)

if invalid_datetime > 0:
    raise ValueError(
        f"{invalid_datetime} timestamps "
        f"could not be parsed."
    )


# ============================================================
# 9. FILTER PROJECT PERIOD
# ============================================================

df = df[
    (df["datetime_utc"] >= START_DATE)
    &
    (df["datetime_utc"] <= END_DATE)
].copy()


# ============================================================
# 10. VALIDATE STATION
# ============================================================

station_codes = sorted(
    df["CODIGO (WMO)"]
    .dropna()
    .astype(str)
    .unique()
)

station_names = sorted(
    df["ESTACAO"]
    .dropna()
    .astype(str)
    .unique()
)

states = sorted(
    df["UF"]
    .dropna()
    .astype(str)
    .unique()
)


print()
print(
    "=== STATION VALIDATION ==="
)

print(
    "WMO codes:",
    station_codes
)

print(
    "Station names:",
    station_names
)

print(
    "States:",
    states
)


if station_codes != [STATION_CODE]:

    raise ValueError(
        f"Unexpected station codes found: "
        f"{station_codes}"
    )


# ============================================================
# 11. SORT AND REMOVE DUPLICATED TIMESTAMPS
# ============================================================

df = (
    df
    .sort_values("datetime_utc")
    .drop_duplicates(
        subset="datetime_utc",
        keep="first",
    )
    .reset_index(drop=True)
)


# ============================================================
# 12. VALIDATE YEAR COVERAGE
# ============================================================

df["year"] = (
    df["datetime_utc"]
    .dt.year
)

year_counts = (
    df["year"]
    .value_counts()
    .sort_index()
)

print()
print(
    "=== OBSERVATIONS BY YEAR ==="
)

display(
    year_counts
    .rename("observations")
    .to_frame()
)


expected_years = set(
    range(2019, 2025)
)

observed_years = set(
    df["year"].unique()
)

missing_years = (
    expected_years
    - observed_years
)

if missing_years:

    raise ValueError(
        f"Missing years in dataset: "
        f"{sorted(missing_years)}"
    )


# ============================================================
# 13. TEMPORAL COVERAGE OF DOWNLOADED FILES
# ============================================================

full_timeline = pd.date_range(
    start="2019-01-01 00:00",
    end="2024-12-31 23:00",
    freq="h",
    tz="UTC",
)

observed_timestamps = pd.DatetimeIndex(
    df["datetime_utc"]
)

missing_timestamps = (
    full_timeline.difference(
        observed_timestamps
    )
)


print()
print(
    "=== CONSOLIDATION SUMMARY ==="
)

print(
    f"Final rows: {len(df)}"
)

print(
    f"First observation: "
    f"{df['datetime_utc'].min()}"
)

print(
    f"Last observation: "
    f"{df['datetime_utc'].max()}"
)

print(
    f"Expected hourly timestamps: "
    f"{len(full_timeline)}"
)

print(
    f"Missing hourly timestamps: "
    f"{len(missing_timestamps)}"
)

print(
    f"Missing percentage: "
    f"{len(missing_timestamps) / len(full_timeline) * 100:.2f}%"
)


# ============================================================
# 14. SAVE FILTERED DATASET
# ============================================================

df = df.drop(
    columns=["year"]
)

df.to_csv(
    OUTPUT_PATH,
    index=False,
)

print()
print(
    f"Dataset saved to: "
    f"{OUTPUT_PATH}"
)

display(
    df.head()
)

Raw files found: 6
- 2019.csv
- 2020.csv
- 2021.csv
- 2022.csv
- 2023.csv
- 2024.csv

Processing file 1/6: 2019.csv
Encoding: utf-8
Separator: ','

Processing file 2/6: 2020.csv
Encoding: utf-8
Separator: ','

Processing file 3/6: 2021.csv
Encoding: utf-8
Separator: ','

Processing file 4/6: 2022.csv
Encoding: utf-8
Separator: ','

Processing file 5/6: 2023.csv
Encoding: utf-8
Separator: ','

Processing file 6/6: 2024.csv
Encoding: utf-8
Separator: ','

Rows read from original files: 27831000
Recife rows collected: 43824

=== STATION VALIDATION ===
WMO codes: ['A301']
Station names: ['RECIFE']
States: ['PE']

=== OBSERVATIONS BY YEAR ===


,observations
year,
2019,8760
2020,8784
2021,8760
2022,8760
2023,8760


ValueError: Missing years in dataset: [2024]

In [3]:
from pathlib import Path
import pandas as pd

FILE_2024 = Path("../data/raw/2024.csv")

encoding, separator = detect_csv_format(FILE_2024)

header = pd.read_csv(
    FILE_2024,
    sep=separator,
    encoding=encoding,
    nrows=0,
)

rename_map = standardize_columns(header.columns)

columns_needed = [
    original
    for original, canonical in rename_map.items()
    if canonical in [
        "Data",
        "CODIGO (WMO)",
        "ESTACAO",
        "UF",
    ]
]

recife_rows = []
recife_by_name = []
total_rows = 0

reader = pd.read_csv(
    FILE_2024,
    sep=separator,
    encoding=encoding,
    usecols=columns_needed,
    chunksize=200_000,
    low_memory=False,
)

for chunk in reader:
    total_rows += len(chunk)

    chunk = chunk.rename(columns=rename_map)

    chunk["CODIGO (WMO)"] = (
        chunk["CODIGO (WMO)"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    chunk["ESTACAO"] = (
        chunk["ESTACAO"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    by_code = chunk[
        chunk["CODIGO (WMO)"] == "A301"
    ]

    if not by_code.empty:
        recife_rows.append(by_code)

    by_name = chunk[
        chunk["ESTACAO"].str.contains(
            "RECIFE",
            na=False
        )
    ]

    if not by_name.empty:
        recife_by_name.append(by_name)


print("=== 2024 DIAGNOSTIC ===")
print("Total rows in file:", total_rows)

if recife_rows:
    df_code = pd.concat(
        recife_rows,
        ignore_index=True
    )

    print()
    print("Rows with WMO A301:", len(df_code))
    print(
        "Station names for A301:",
        df_code["ESTACAO"].unique()
    )
    print(
        "States for A301:",
        df_code["UF"].unique()
    )
    print(
        "First dates:",
        df_code["Data"].head().tolist()
    )
    print(
        "Last dates:",
        df_code["Data"].tail().tolist()
    )

else:
    print()
    print("No rows found with WMO code A301.")


if recife_by_name:
    df_name = pd.concat(
        recife_by_name,
        ignore_index=True
    )

    print()
    print("Rows whose station name contains RECIFE:", len(df_name))
    print(
        "WMO codes associated with RECIFE:",
        df_name["CODIGO (WMO)"].unique()
    )
    print(
        "Station names:",
        df_name["ESTACAO"].unique()
    )

else:
    print()
    print("No station containing RECIFE was found.")

=== 2024 DIAGNOSTIC ===
Total rows in file: 2472288

No rows found with WMO code A301.

No station containing RECIFE was found.


In [6]:
from pathlib import Path
import zipfile
from collections import Counter


RAW_DIR = Path("../data/raw")

ZIP_PATH = RAW_DIR / "2024_inmet.zip"


if not ZIP_PATH.exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {ZIP_PATH}"
    )


# ============================================================
# 1. OPEN ZIP
# ============================================================

with zipfile.ZipFile(ZIP_PATH, "r") as archive:

    members = [
        member
        for member in archive.infolist()
        if not member.is_dir()
    ]

    print("=== ZIP STRUCTURE ===")
    print("Files inside ZIP:", len(members))

    extensions = Counter(
        Path(member.filename).suffix.lower()
        for member in members
    )

    print("File types:")
    for extension, count in extensions.items():
        print(
            extension or "[no extension]",
            ":",
            count
        )


    # ========================================================
    # 2. SEARCH INSIDE CSV CONTENT
    # ========================================================

    matches = []

    csv_members = [
        member
        for member in members
        if member.filename.lower().endswith(".csv")
    ]

    print()
    print("CSV files found:", len(csv_members))
    print()
    print("Searching CSV contents for A301 / RECIFE...")


    for position, member in enumerate(
        csv_members,
        start=1
    ):

        # INMET station identification is normally
        # located near the beginning of each file.
        with archive.open(member) as file:

            beginning = file.read(64_000)

        beginning_upper = beginning.upper()

        if (
            b"A301" in beginning_upper
            or b"RECIFE" in beginning_upper
        ):
            matches.append(member.filename)

            print()
            print("MATCH FOUND:")
            print(member.filename)


        if position % 100 == 0:
            print(
                f"Checked {position}/{len(csv_members)} files..."
            )


# ============================================================
# 3. RESULT
# ============================================================

print()
print("=== SEARCH RESULT ===")
print("Matches found:", len(matches))

if matches:

    print()
    print("Files associated with Recife/A301:")

    for match in matches:
        print("-", match)

else:

    print()
    print(
        "A301 / RECIFE was not found "
        "inside the beginning of any CSV."
    )

=== ZIP STRUCTURE ===
Files inside ZIP: 565
File types:
.csv : 565

CSV files found: 565

Searching CSV contents for A301 / RECIFE...
Checked 100/565 files...
Checked 200/565 files...
Checked 300/565 files...
Checked 400/565 files...
Checked 500/565 files...

=== SEARCH RESULT ===
Matches found: 0

A301 / RECIFE was not found inside the beginning of any CSV.


## 2024 Data Availability

The original project scope considered meteorological data from 2018 to 2024.

During data acquisition and validation, the Recife automatic weather station (WMO A301) was not found in the official INMET historical data package for 2024.

The package was inspected both by file names and by searching the contents of all 565 CSV files, and no records associated with `A301` or `RECIFE` were identified.

To preserve spatial and instrumental consistency, data from another weather station will not be used as a replacement.

Therefore, the current modeling dataset will use observations from the Recife A301 station for the period **2018–2023**.

The absence of 2024 data will be documented as a limitation of the available dataset.